## Gold Layer — Recent Activities Enriched

Enriched view of the last 90 days of activities with full context for training coach AI recommendations.

**Purpose:**
* Contextual workout suggestions based on recent patterns
* Equipment and weather-based insights
* Day/time pattern analysis
* Recovery and effort tracking

**Source:** `garmin_lakehouse.silver.activity_details`

**Refresh Cadence:** Daily (or more frequently for real-time coaching)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.gold;

In [0]:
%sql
DROP TABLE IF EXISTS garmin_lakehouse.gold.running_activities;

CREATE OR REPLACE VIEW garmin_lakehouse.gold.running_activities AS
SELECT
    -- identifiers
    activity_id,
    owner_id,                                -- for when your partner's data lands
    activity_name,
    description,                             -- raw; workout_type gets parsed from this later
    activity_type_key,

    -- timing (local)
    start_date,
    start_time_local,
    start_hour,
    DATE_FORMAT(start_date, 'EEEE') AS day_name,
    CASE
        WHEN start_hour < 6  THEN 'Early Morning'
        WHEN start_hour < 12 THEN 'Morning'
        WHEN start_hour < 17 THEN 'Afternoon'
        WHEN start_hour < 21 THEN 'Evening'
        ELSE 'Night'
    END AS time_of_day_label,
    DATEDIFF(CURRENT_DATE(), start_date) AS days_ago,   -- safe: it's a VIEW, recomputed each query

    -- distance & duration
    distance_km,
    duration_minutes,
    moving_duration_minutes,

    -- pace & speed
    avg_pace_min_per_km,
    avg_pace_formatted,
    avg_speed_kmh,

    -- heart rate (raw facts — the app derives effort)
    avg_heart_rate,
    max_heart_rate,
    hr_zone_1_seconds,
    hr_zone_2_seconds,
    hr_zone_3_seconds,
    hr_zone_4_seconds,
    hr_zone_5_seconds,

    -- running form
    avg_cadence_spm,
    avg_stride_length_m,

    -- training load (facts)
    aerobic_training_effect,
    anaerobic_training_effect,
    vo2_max,

    -- elevation
    elevation_gain_m,
    elevation_loss_m,

    -- gear context (already joined in activity_details)
    gear_name,
    gear_type,
    gear_status,
    has_gear,

    -- weather context (already joined in activity_details)
    temperature_c,
    feels_like_c,
    wind_speed_kmh,
    weather_condition,
    has_weather,

    -- other
    total_calories,
    loaded_at
FROM garmin_lakehouse.silver.activity_details
WHERE activity_type_key LIKE '%running%'

In [0]:
%sql
select count(*) from garmin_lakehouse.gold.running_activities